*A user-guided Python notebook for retrieving, annotating, curating and processing GEO microarray expression datasets.*

## How to use this notebook

This notebook retrieves and processes GEO microarray gene-expression datasets for downstream cross-dataset integration and analysis.

**Instructions**
1. Run the cells sequentially from **Step 1 to Step 19**.
2. In **Step 3**, enter the required GEO Series accession number when prompted (e.g., `GSE75214`).
3. After the phenotype metadata are displayed, use **Step 9** to select the column that most clearly contains the Disease/Case and Control/Healthy information.
4. **Step 12** will automatically display the unique phenotype groups from the selected column.
5. In **Step 13**, enter any unwanted sample group(s) that should be removed. Leave the input blank if all groups should be retained.
6. In **Step 14**, classify each retained group as **Disease/Case** or **Control/Healthy** when prompted.
7. Run the remaining cells sequentially. The final cleaned gene-expression dataset will be downloaded automatically.

**Important:** GEO datasets may differ in platform annotation and phenotype metadata. Users should verify the selected platform, phenotype column, sample groups, and Disease/Control assignments before generating the final output.

**Developed by:** Enamul Haq Rana and Md Fazlay Rabbi  
Department of Zoology, University of Rajshahi, Bangladesh.

In [1]:
# @title # Step 1: Install required packages (Just run the cell)
# This keeps the original workflow and installs only the libraries needed for GEO microarray processing.
!pip install -q GEOparse pandas numpy matplotlib seaborn mygene

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.3 MB/s eta 0:00:00


In [2]:
# @title # Step 2: Import necessary libraries (Just run the cell)
import os
import re
import warnings

import pandas as pd
import numpy as np
import GEOparse
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

warnings.filterwarnings("ignore", category=FutureWarning)


# =============================================================================
# SMALL HELPER FUNCTIONS
# These functions only make the original cells compatible with different GPL
# annotation structures. They do not change the workflow.
# =============================================================================

def normalize_column_name(name):
    """Normalize a column name for safe platform-independent matching."""
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_annotation_column(columns, exact_candidates=None, token_groups=None):
    """Find an annotation column across Affymetrix, Illumina, Agilent, or other GPL tables."""
    columns = list(columns)
    normalized = {col: normalize_column_name(col) for col in columns}

    for candidate in exact_candidates or []:
        candidate_norm = normalize_column_name(candidate)
        for col, col_norm in normalized.items():
            if col_norm == candidate_norm:
                return col

    for tokens in token_groups or []:
        token_norms = [normalize_column_name(token) for token in tokens]
        for col, col_norm in normalized.items():
            if all(token in col_norm for token in token_norms):
                return col

    return None


def flatten_metadata_value(value):
    """Convert GEO metadata lists into readable text without losing information."""
    if isinstance(value, (list, tuple)):
        cleaned = [str(item).strip() for item in value if str(item).strip()]
        return " | ".join(cleaned)
    if value is None:
        return np.nan
    return value


def get_gsm_platform_id(gsm):
    """Return the GPL accession used by one GSM sample."""
    value = gsm.metadata.get("platform_id", [])
    if isinstance(value, (list, tuple)):
        return str(value[0]).strip() if value else ""
    return str(value).strip()


def clean_gene_symbol(value):
    """Clean a direct gene-symbol annotation while retaining the original first-symbol rule."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text or text.lower() in {"nan", "na", "n/a", "none", "null", "---", "-"}:
        return np.nan

    # The original workflow used the first symbol before ' /// '.
    text = re.split(r"\s*///\s*", text)[0].strip()

    # Some GPLs separate alternatives with semicolons or pipes.
    text = re.split(r"\s*[;|]\s*", text)[0].strip()

    if not text or " " in text:
        return np.nan

    if re.fullmatch(r"[A-Za-z][A-Za-z0-9._-]*", text):
        return text.upper()

    return np.nan


def extract_symbol_from_assignment(value):
    """Extract a likely human gene symbol from Affymetrix/Agilent assignment text."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text:
        return np.nan

    tokens = [
        token.strip()
        for token in re.split(r"\s*///\s*|\s*//\s*|\s*[;|]\s*", text)
        if token.strip()
    ]

    accession_prefixes = (
        "NM_", "NR_", "XM_", "XR_", "ENSG", "ENST", "ENSP",
        "ILMN_", "A_", "AFFX", "GI:", "GB:", "REFSEQ:", "ENTREZ"
    )

    for token in tokens:
        token = token.strip("[](){}\"'")
        if token.upper().startswith(accession_prefixes):
            continue
        if re.fullmatch(r"[A-Za-z][A-Za-z0-9._-]{1,30}", token) and " " not in token:
            return token.upper()

    return np.nan


def first_identifier(value):
    """Return the first Entrez or RefSeq identifier from a multi-value annotation."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text.lower() in {"nan", "na", "n/a", "none", "null", "---", "-"}:
        return np.nan
    return re.split(r"\s*///\s*|\s*//\s*|\s*[;,|]\s*", text)[0].strip()


def map_identifiers_with_mygene(values, scopes, species="human"):
    """Optional fallback when a GPL supplies Entrez/RefSeq IDs but no gene-symbol column."""
    identifiers = pd.Series(values).dropna().map(first_identifier).dropna().astype(str)
    identifiers = identifiers[identifiers.str.strip().ne("")].drop_duplicates().tolist()
    if not identifiers:
        return {}

    try:
        import mygene
        mg = mygene.MyGeneInfo()
        mapping = {}

        for start in range(0, len(identifiers), 1000):
            batch = identifiers[start:start + 1000]
            results = mg.querymany(
                batch,
                scopes=scopes,
                fields="symbol",
                species=species,
                as_dataframe=False,
                verbose=False,
            )
            for result in results:
                query = str(result.get("query", "")).strip()
                symbol = clean_gene_symbol(result.get("symbol"))
                if query and pd.notna(symbol):
                    mapping.setdefault(query, symbol)

        return mapping

    except Exception as error:
        print(f"MyGene fallback warning: {error}")
        return {}

In [4]:
# @title Step 3: Download the selected GEO microarray dataset
# Run this cell and enter the required GEO Series accession when prompted.

# Ask the user for the GEO accession number
GSE_ID = input(
    "Enter the GEO Series accession number (example: GSE75214, GSE19420, GSE66407): "
).strip().upper()

# Basic validation
while not (GSE_ID.startswith("GSE") and GSE_ID[3:].isdigit()):
    print("Invalid GEO accession. Please enter a valid GSE number, e.g., GSE75214.")
    GSE_ID = input(
        "Enter the GEO Series accession number: "
    ).strip().upper()

# Default settings
PLATFORM_ID = None
USE_MYGENE_FALLBACK = True
DOWNLOAD_INTERMEDIATE_FILES = False

print(f"\nSelected dataset: {GSE_ID}")
print(f"Downloading {GSE_ID} from NCBI GEO...")

gse = GEOparse.get_GEO(
    geo=GSE_ID,
    destdir="./",
    silent=False
)

print("\nDataset downloaded successfully!")
print("Available platforms:", list(gse.gpls.keys()))

Enter the GEO Series accession number (example: GSE75214, GSE19420, GSE66407): GSE75214


20-Oct-2026 05:37:32 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
20-Oct-2026 05:37:32 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
20-Oct-2026 05:37:32 INFO GEOparse - Parsing ./GSE75214_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE75214_family.soft.gz: 
20-Oct-2026 05:37:32 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
20-Oct-2026 05:37:32 DEBUG GEOparse - SERIES: GSE75214
DEBUG:GEOparse:SERIES: GSE75214
20-Oct-2026 05:37:32 DEBUG GEOparse - PLATFORM: GPL6244
DEBUG:GEOparse:PLATFORM: GPL6244



Selected dataset: GSE75214


20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426071
DEBUG:GEOparse:SAMPLE: GSM1426071
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426072
DEBUG:GEOparse:SAMPLE: GSM1426072
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426073
DEBUG:GEOparse:SAMPLE: GSM1426073
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426074
DEBUG:GEOparse:SAMPLE: GSM1426074
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426075
DEBUG:GEOparse:SAMPLE: GSM1426075
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426076
DEBUG:GEOparse:SAMPLE: GSM1426076
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426077
DEBUG:GEOparse:SAMPLE: GSM1426077
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426078
DEBUG:GEOparse:SAMPLE: GSM1426078
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426079
DEBUG:GEOparse:SAMPLE: GSM1426079
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426080
DEBUG:GEOparse:SAMPLE: GSM1426080
20-Oct-2026 05:37:34 DEBUG GEOparse - SAMPLE: GSM1426081
DEBUG:GEOparse:SAMPLE: GSM1426081


Dataset downloaded successfully!
Available platforms: ['GPL6244']


In [5]:
# @title # Step 4: Extract the expression data (Just run the cell)
print("\nStep 4: Extracting expression data...")

# Keep the original idea of selecting one platform, but make the selection safe
# when a GEO series contains more than one GPL.
platform_sample_counts = {}
for gsm_name, gsm in gse.gsms.items():
    gsm_platform = get_gsm_platform_id(gsm)
    platform_sample_counts[gsm_platform] = platform_sample_counts.get(gsm_platform, 0) + 1

if PLATFORM_ID is None:
    valid_platform_counts = {
        key: value
        for key, value in platform_sample_counts.items()
        if key in gse.gpls
    }
    if valid_platform_counts:
        platform_id = max(valid_platform_counts, key=valid_platform_counts.get)
    else:
        platform_id = list(gse.gpls.keys())[0]
else:
    platform_id = str(PLATFORM_ID)

if platform_id not in gse.gpls:
    raise ValueError(
        f"Selected platform {platform_id} is not available. "
        f"Available platforms: {list(gse.gpls.keys())}"
    )

gpl = platform_id
print("Selected platform:", platform_id)
print("Samples per platform:", platform_sample_counts)

# Get expression data from the dataset
expression_set = pd.DataFrame()
processed_sample_ids = []
skipped_samples = []

# Process each sample exactly as in the original workflow
for gsm_name, gsm in gse.gsms.items():
    gsm_platform = get_gsm_platform_id(gsm)
    if gsm_platform and gsm_platform != platform_id:
        continue

    table = gsm.table.copy()

    id_ref_col = find_annotation_column(
        table.columns,
        exact_candidates=["ID_REF", "ID", "Probe ID", "ProbeID", "SPOT_ID"],
        token_groups=[["id", "ref"], ["probe", "id"]],
    )
    value_col = find_annotation_column(
        table.columns,
        exact_candidates=["VALUE", "Signal", "Normalized Signal", "Processed Signal"],
        token_groups=[["value"], ["signal"]],
    )

    if id_ref_col is None or value_col is None:
        skipped_samples.append({
            "sample_id": gsm_name,
            "reason": f"ID/value columns not detected; columns={table.columns.tolist()}"
        })
        continue

    sample_data = table[[id_ref_col, value_col]].copy()
    sample_data.columns = ["ID_REF", gsm_name]
    sample_data["ID_REF"] = sample_data["ID_REF"].astype(str).str.strip()
    sample_data[gsm_name] = pd.to_numeric(sample_data[gsm_name], errors="coerce")
    sample_data = sample_data.drop_duplicates(subset="ID_REF", keep="first")

    # If this is the first sample, use it as the base dataframe
    if expression_set.empty:
        expression_set = sample_data
    else:
        # Merge with existing data, preserving the original outer-merge logic
        expression_set = pd.merge(expression_set, sample_data, on="ID_REF", how="outer")

    processed_sample_ids.append(gsm_name)

if expression_set.empty:
    raise RuntimeError(
        "No expression data could be extracted. Inspect the GSM table columns and selected platform."
    )

# Rename ID_REF to ID for consistency with the original workflow
expression_set.rename(columns={"ID_REF": "ID"}, inplace=True)

print("Expression data shape:", expression_set.shape)
print("Processed samples:", len(processed_sample_ids))
print("Skipped samples:", len(skipped_samples))
if skipped_samples:
    display(pd.DataFrame(skipped_samples).head())

print("Expression data preview:")
display(expression_set.head())


Step 4: Extracting expression data...
Selected platform: GPL6244
Samples per platform: {'GPL6244': 194}
Expression data shape: (33252, 195)
Processed samples: 194
Skipped samples: 0
Expression data preview:


,ID,GSM1426071,GSM1426072,GSM1426073,GSM1426074,GSM1426075,GSM1426076,GSM1426077,GSM1426078,GSM1426079,...,GSM1945827,GSM1945828,GSM1945829,GSM1945830,GSM1945831,GSM1945832,GSM1945833,GSM1945834,GSM1945835,GSM1945836
0,7892501,3.097244,4.459059,3.322385,4.184156,3.567690,4.434777,3.416537,3.286221,2.897307,...,3.068425,3.508979,2.937225,3.404894,4.506227,2.637388,4.403679,3.310945,3.651002,3.141966
1,7892502,4.622620,4.866747,5.793462,5.652146,5.106890,5.180421,5.198954,5.169658,5.014645,...,4.957102,5.289645,4.914546,5.476374,5.059218,4.862137,5.287762,5.010893,5.194320,5.077890
2,7892503,4.588315,4.635416,7.472632,6.024110,4.213697,4.001750,4.180109,4.076452,4.189199,...,4.373730,4.926381,4.039336,4.759516,4.487999,4.315397,4.510607,4.947265,4.357883,4.655735
3,7892504,9.635181,8.897930,9.450976,8.800208,9.105323,9.350480,9.066997,8.958961,8.896822,...,8.757489,9.303847,9.013299,9.003382,9.400929,9.672492,9.295140,9.452490,9.183286,9.279613
4,7892505,4.524358,4.131027,4.633418,4.228925,4.757025,4.703955,3.916718,4.552928,4.238462,...,5.027381,4.365907,3.488814,4.337687,5.196698,4.058416,4.535763,4.855737,4.711060,4.523698


In [ ]:
 # @title Optional check: Inspect platform annotation features (Run the cell If Nan values are seen above the cell)
# ORIGINAL CUSTOM INSPECTION STEP RETAINED.

# It is now safe for Illumina, Affymetrix, Agilent, and other GPL annotation tables.

# Safety initialization fixes the original NameError without moving this cell.
if "platform_id" not in globals():
    platform_id = PLATFORM_ID if PLATFORM_ID is not None else list(gse.gpls.keys())[0]
if "feature_data" not in globals():
    feature_data = gse.gpls[platform_id].table.copy()

probe_id_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["ID", "ID_REF", "Probe ID", "ProbeID", "Probe_Id", "SPOT_ID", "NAME"],
    token_groups=[["probe", "id"], ["spot", "id"]],
)
symbol_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=[
        "Symbol", "Gene Symbol", "Gene symbol", "GENE_SYMBOL", "GeneSymbol",
        "GENE", "ILMN_Gene", "Gene_Symbol", "SYMBOL"
    ],
    token_groups=[["gene", "symbol"]],
)
entrez_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Entrez_Gene_ID", "ENTREZ_GENE_ID", "Gene ID", "GENE_ID", "Entrez Gene"],
    token_groups=[["entrez"], ["gene", "id"]],
)
refseq_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["RefSeq_ID", "REFSEQ", "RefSeq", "RefSeq Transcript ID"],
    token_groups=[["refseq"]],
)
probe_type_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Probe_Type", "CONTROL_TYPE", "Control Type", "SPOT_TYPE"],
    token_groups=[["probe", "type"], ["control", "type"]],
)
chromosome_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Chromosome", "CHR", "Chromosomal Location"],
    token_groups=[["chromosome"], ["chromosomal", "location"]],
)
definition_preview_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=["Definition", "Gene title", "Gene Title", "GENE_NAME", "Description"],
    token_groups=[["gene", "title"], ["description"]],
)

# Keep biological probes when at least one useful biological annotation is present.
annotation_presence = pd.Series(False, index=feature_data.index)
for selected_col in [symbol_preview_col, entrez_preview_col, refseq_preview_col]:
    if selected_col is not None:
        valid = feature_data[selected_col].notna() & feature_data[selected_col].astype(str).str.strip().ne("")
        annotation_presence = annotation_presence | valid

# If the platform has no standard biological-annotation columns, retain all rows for inspection.
if not annotation_presence.any():
    annotation_presence = pd.Series(True, index=feature_data.index)

# Exclude obvious technical controls when the GPL supplies a control/probe-type column.
if probe_type_preview_col is not None:
    technical_control = feature_data[probe_type_preview_col].astype(str).str.contains(
        r"control|spike|blank|negative|positive|housekeeping", case=False, regex=True, na=False
    )
else:
    technical_control = pd.Series(False, index=feature_data.index)

biological_probes = feature_data.loc[annotation_presence & ~technical_control].copy()

# Keep the original variable name 'illumina_features' so the user's workflow remains familiar.
preview_columns = [
    probe_id_preview_col,
    symbol_preview_col,
    entrez_preview_col,
    refseq_preview_col,
    probe_type_preview_col,
    chromosome_preview_col,
    definition_preview_col,
]
preview_columns = [col for col in preview_columns if col is not None]
preview_columns = list(dict.fromkeys(preview_columns))

illumina_features = (
    biological_probes[preview_columns].copy()
    if preview_columns
    else biological_probes.copy()
)

print("Filtered Microarray Features (Biological Probes Only):")
display(illumina_features.head())
print("\nShape after filtering:", illumina_features.shape)
print("Detected preview columns:", preview_columns)

Filtered Microarray Features (Biological Probes Only):


,ID
0,7896736
1,7896738
2,7896740
3,7896742
4,7896744



Shape after filtering: (33297, 1)
Detected preview columns: ['ID']


In [6]:
# @title # Step 5: Get feature data (probe annotations) (Just run the cell)
print("\nStep 5: Getting feature data...")

# Re-read the same selected GPL, retaining the original Step 5 inspection cell.
if "platform_id" not in globals():
    platform_id = PLATFORM_ID if PLATFORM_ID is not None else list(gse.gpls.keys())[0]

feature_data = gse.gpls[platform_id].table.copy()

# Display the feature data columns to verify structure
print("Platform ID:", platform_id)
print("Feature data shape:", feature_data.shape)
print("Feature data columns:", feature_data.columns.tolist())
print("Feature data preview:")
display(feature_data.head())


Step 5: Getting feature data...
Platform ID: GPL6244
Feature data shape: (33297, 12)
Feature data columns: ['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment', 'mrna_assignment', 'category']
Feature data preview:


,ID,GB_LIST,SPOT_ID,seqname,RANGE_GB,RANGE_STRAND,RANGE_START,RANGE_STOP,total_probes,gene_assignment,mrna_assignment,category
0,7896736,NaN,chr1:53049-54936,chr1,NC_000001.10,+,53049,54936,7,---,NONHSAT060105 // NONCODE // Non-coding transcr...,main
1,7896738,NaN,chr1:63015-63887,chr1,NC_000001.10,+,63015,63887,31,ENST00000328113 // OR4G2P // olfactory recepto...,ENST00000328113 // ENSEMBL // havana:known chr...,main
2,7896740,"NM_001004195,NM_001005240,NM_001005484,BC13684...",chr1:69091-70008,chr1,NC_000001.10,+,69091,70008,24,"NM_001004195 // OR4F4 // olfactory receptor, f...",NM_001004195 // RefSeq // Homo sapiens olfacto...,main
3,7896742,"NR_024437,XM_006711854,XM_006726377,XR_430662,...",chr1:334129-334296,chr1,NC_000001.10,+,334129,334296,6,NR_024437 // LOC728323 // uncharacterized LOC7...,NR_024437 // RefSeq // Homo sapiens uncharacte...,main
4,7896744,"NM_001005221,NM_001005224,NM_001005277,NM_0010...",chr1:367659-368597,chr1,NC_000001.10,+,367659,368597,36,"NM_001005221 // OR4F29 // olfactory receptor, ...",NM_001005221 // RefSeq // Homo sapiens olfacto...,main


In [7]:
# @title # Step 6: Check for correct column names for gene symbol (Just run the cell)
print("\nStep 6: Checking column names for gene symbol...")

# Editable candidate lists are intentionally visible so this step can still be customized.
possible_probe_id_cols = [
    "ID", "ID_REF", "Probe ID", "ProbeID", "Probe_Id", "PROBE_ID",
    "SPOT_ID", "NAME", "Reporter Identifier", "Array_Address_Id"
]

possible_gene_symbol_cols = [
    "gene_symbol", "Gene symbol", "Gene Symbol", "Symbol", "SYMBOL",
    "gene symbol", "GENE_SYMBOL", "GeneSymbol", "Gene_Symbol",
    "ILMN_Gene", "GENE"
]

possible_entrez_cols = [
    "Entrez_Gene_ID", "ENTREZ_GENE_ID", "Entrez Gene", "EntrezGene ID",
    "Gene ID", "GENE_ID", "ENTREZID"
]

possible_refseq_cols = [
    "RefSeq_ID", "REFSEQ", "RefSeq", "RefSeq Transcript ID", "RefSeq_IDs"
]

possible_assignment_cols = [
    "gene_assignment", "Gene Assignment", "Gene assignment",
    "Target Description", "TARGET_DESCRIPTION", "Description"
]

possible_control_cols = [
    "Probe_Type", "CONTROL_TYPE", "Control Type", "SPOT_TYPE", "ControlType"
]

probe_id_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_probe_id_cols,
    token_groups=[["probe", "id"], ["spot", "id"], ["reporter", "identifier"]],
)

gene_symbol_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_gene_symbol_cols,
    token_groups=[["gene", "symbol"]],
)

entrez_gene_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_entrez_cols,
    token_groups=[["entrez"], ["gene", "id"]],
)

refseq_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_refseq_cols,
    token_groups=[["refseq"]],
)

gene_assignment_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_assignment_cols,
    token_groups=[["gene", "assignment"], ["target", "description"]],
)

control_probe_col = find_annotation_column(
    feature_data.columns,
    exact_candidates=possible_control_cols,
    token_groups=[["control", "type"], ["probe", "type"]],
)

if probe_id_col is None:
    raise ValueError(
        "Could not identify the probe-ID column. Available columns are: "
        + str(feature_data.columns.tolist())
    )

print("Detected probe ID column       :", probe_id_col)
print("Detected gene symbol column    :", gene_symbol_col)
print("Detected Entrez ID column      :", entrez_gene_col)
print("Detected RefSeq column         :", refseq_col)
print("Detected assignment column     :", gene_assignment_col)
print("Detected control-probe column  :", control_probe_col)

if gene_symbol_col is None:
    print(
        "No direct gene-symbol column was found. Step 6 will use assignment text, "
        "Entrez ID, or RefSeq ID as a controlled fallback."
    )


Step 6: Checking column names for gene symbol...
Detected probe ID column       : ID
Detected gene symbol column    : None
Detected Entrez ID column      : None
Detected RefSeq column         : None
Detected assignment column     : gene_assignment
Detected control-probe column  : None
No direct gene-symbol column was found. Step 6 will use assignment text, Entrez ID, or RefSeq ID as a controlled fallback.


In [8]:
# @title # Step 7: Create gene_info dataframe (Just run the cell)
print("\nStep 7: Creating gene information table...")

# Retain the original gene_info structure: probe_id, gene_symbol, gene_id.
annotation_source = feature_data.copy()

gene_info = pd.DataFrame({
    "probe_id": annotation_source[probe_id_col].astype(str).str.strip(),
    "gene_symbol": (
        annotation_source[gene_symbol_col]
        if gene_symbol_col is not None
        else pd.Series(pd.NA, index=annotation_source.index, dtype="object")
    ),
    "gene_id": annotation_source[probe_id_col].astype(str).str.strip(),
})

# Keep optional source columns temporarily for fallback mapping.
if entrez_gene_col is not None:
    gene_info["_entrez"] = annotation_source[entrez_gene_col]
else:
    gene_info["_entrez"] = np.nan

if refseq_col is not None:
    gene_info["_refseq"] = annotation_source[refseq_col]
else:
    gene_info["_refseq"] = np.nan

if gene_assignment_col is not None:
    gene_info["_assignment"] = annotation_source[gene_assignment_col]
else:
    gene_info["_assignment"] = np.nan

# Clean direct symbols using the same first-symbol principle as the original code.
gene_info["gene_symbol"] = gene_info["gene_symbol"].map(clean_gene_symbol)

# Affymetrix and some Agilent GPLs store symbols inside assignment text.
missing_symbol = gene_info["gene_symbol"].isna()
if missing_symbol.any() and gene_assignment_col is not None:
    gene_info.loc[missing_symbol, "gene_symbol"] = (
        gene_info.loc[missing_symbol, "_assignment"].map(extract_symbol_from_assignment)
    )

# Optional fallback for GPLs that contain Entrez IDs but no usable symbol column.
missing_symbol = gene_info["gene_symbol"].isna()
if USE_MYGENE_FALLBACK and missing_symbol.any() and entrez_gene_col is not None:
    print("Mapping unresolved Entrez IDs to gene symbols using MyGene...")
    entrez_map = map_identifiers_with_mygene(
        gene_info.loc[missing_symbol, "_entrez"],
        scopes="entrezgene",
        species="human",
    )
    first_entrez = gene_info["_entrez"].map(first_identifier)
    gene_info.loc[missing_symbol, "gene_symbol"] = first_entrez.loc[missing_symbol].map(entrez_map)

# Optional RefSeq fallback if required.
missing_symbol = gene_info["gene_symbol"].isna()
if USE_MYGENE_FALLBACK and missing_symbol.any() and refseq_col is not None:
    print("Mapping unresolved RefSeq IDs to gene symbols using MyGene...")
    refseq_map = map_identifiers_with_mygene(
        gene_info.loc[missing_symbol, "_refseq"],
        scopes="refseq",
        species="human",
    )
    first_refseq = gene_info["_refseq"].map(first_identifier)
    gene_info.loc[missing_symbol, "gene_symbol"] = first_refseq.loc[missing_symbol].map(refseq_map)

# Exclude obvious technical controls only when the GPL provides an explicit control column.
if control_probe_col is not None:
    technical_control = annotation_source[control_probe_col].astype(str).str.contains(
        r"control|spike|blank|negative|positive|housekeeping",
        case=False,
        regex=True,
        na=False,
    )
    gene_info = gene_info.loc[~technical_control].copy()

# Remove rows with missing gene symbols or probe IDs.
gene_info["gene_symbol"] = gene_info["gene_symbol"].map(clean_gene_symbol)
gene_info = gene_info[
    gene_info["gene_symbol"].notna()
    & gene_info["gene_symbol"].astype(str).str.strip().ne("")
    & gene_info["probe_id"].notna()
    & gene_info["probe_id"].astype(str).str.strip().ne("")
].copy()

gene_info = gene_info.drop_duplicates(subset="probe_id", keep="first")
gene_info = gene_info[["probe_id", "gene_symbol", "gene_id"]]

print("Gene info shape:", gene_info.shape)
print("Gene info preview:")
display(gene_info.head())


Step 6: Creating gene information table...
Gene info shape: (25293, 3)
Gene info preview:


,probe_id,gene_symbol,gene_id
1,7896738,OR4G2P,7896738
2,7896740,OR4F4,7896740
3,7896742,LOC728323,7896742
4,7896744,OR4F29,7896744
5,7896746,MT-TM,7896746


In [9]:
# @title # Step 8: Create phenotype dataframe with full metadata (Just run the cell)
# ORIGINAL PHENOTYPE INSPECTION STEP RETAINED.

phenotype_rows = []
sample_ids_for_metadata = processed_sample_ids if "processed_sample_ids" in globals() else list(gse.gsms.keys())

for sample_id in sample_ids_for_metadata:
    gsm = gse.gsms[sample_id]
    metadata = {key: flatten_metadata_value(value) for key, value in gsm.metadata.items()}

    characteristics = gsm.metadata.get("characteristics_ch1", [])
    if not isinstance(characteristics, (list, tuple)):
        characteristics = [characteristics]

    row = {
        "sample_id": gsm.name,
        **metadata,
        **{
            f"characteristics_{i}": flatten_metadata_value(char)
            for i, char in enumerate(characteristics)
        },
    }
    phenotype_rows.append(row)

pheno_data = pd.DataFrame(phenotype_rows)
pheno_data.reset_index(drop=True, inplace=True)

print("Phenotype data shape:", pheno_data.shape)
print("Phenotype data columns:")
print(pheno_data.columns.tolist())
print("Phenotype data preview:")
display(pheno_data.head())

# Save the dataframe exactly as in the original workflow.
pheno_data.to_csv("pheno_data.csv", index=False)

# Keep False when only the final processed CSV should be downloaded.
if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("pheno_data.csv")

Phenotype data shape: (194, 36)
Phenotype data columns:
['sample_id', 'title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_laboratory', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count', 'characteristics_0', 'characteristics_1', 'characteristics_2']
Phenotype data preview:


,sample_id,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_address,contact_city,contact_zip/postal_code,contact_country,supplementary_file,series_id,data_row_count,characteristics_0,characteristics_1,characteristics_2
0,GSM1426071,CD_colon_active_1,GSM1426071,Public on May 27 2015,Jul 03 2014,May 27 2015,RNA,1,Biopsy from inflamed colonic mucosa of active ...,Homo sapiens,...,Herestraat 49,Leuven,3000,Belgium,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1426...,GSE59071 | GSE75214,33252,tissue: colon,disease: Crohn's disease,disease activity: active
1,GSM1426072,CD_colon_active_2,GSM1426072,Public on May 27 2015,Jul 03 2014,May 27 2015,RNA,1,Biopsy from inflamed colonic mucosa of active ...,Homo sapiens,...,Herestraat 49,Leuven,3000,Belgium,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1426...,GSE59071 | GSE75214,33252,tissue: colon,disease: Crohn's disease,disease activity: active
2,GSM1426073,CD_colon_active_3,GSM1426073,Public on May 27 2015,Jul 03 2014,May 27 2015,RNA,1,Biopsy from inflamed colonic mucosa of active ...,Homo sapiens,...,Herestraat 49,Leuven,3000,Belgium,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1426...,GSE59071 | GSE75214,33252,tissue: colon,disease: Crohn's disease,disease activity: active
3,GSM1426074,CD_colon_active_4,GSM1426074,Public on May 27 2015,Jul 03 2014,May 27 2015,RNA,1,Biopsy from inflamed colonic mucosa of active ...,Homo sapiens,...,Herestraat 49,Leuven,3000,Belgium,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1426...,GSE59071 | GSE75214,33252,tissue: colon,disease: Crohn's disease,disease activity: active
4,GSM1426075,CD_colon_active_5,GSM1426075,Public on May 27 2015,Jul 03 2014,May 27 2015,RNA,1,Biopsy from inflamed colonic mucosa of active ...,Homo sapiens,...,Herestraat 49,Leuven,3000,Belgium,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1426...,GSE59071 | GSE75214,33252,tissue: colon,disease: Crohn's disease,disease activity: active


In [13]:
# @title Step 9: Select the phenotype description column
# Run this cell and enter the column name containing the Case/Control/Healthy information.

print("\nStep 9: Selecting the phenotype description column...")

print(
    "\nIMPORTANT:\n"
    "Check the phenotype table displayed in the previous cell.\n"
    "Identify the column that most clearly contains the sample groups, "
    "such as Case/Disease and Control/Healthy.\n"
)

# Show available columns to make selection easier
print("Available phenotype columns:")
for col in pheno_data.columns:
    print(f" - {col}")

# Ask the user for the desired phenotype column
description_column = input(
    "\nEnter the phenotype column name "
    "(example: characteristics_1): "
).strip()

# Keep asking until a valid and non-empty column is provided
while (
    description_column not in pheno_data.columns
    or pheno_data[description_column].dropna().empty
):
    print(
        f"\n'{description_column}' is not a valid or usable column."
    )
    print(
        "Please check the phenotype table above and enter the exact column name "
        "that contains Case/Control/Healthy or disease-status information."
    )

    description_column = input(
        "\nEnter the phenotype column name: "
    ).strip()

selected_description_column = description_column

# Create sample information table
sample_info = pd.DataFrame({
    "geo_accession": pheno_data["sample_id"].astype(str),
    "description": pheno_data[selected_description_column].map(flatten_metadata_value),
})

print(f"\nSelected phenotype column: {selected_description_column}")

print("\nUnique phenotype descriptions found in this column:")
for value in sample_info["description"].dropna().unique():
    print(f" - {value}")

print("\nSample information preview:")
display(sample_info.head())


Step 9: Selecting the phenotype description column...

IMPORTANT:
Check the phenotype table displayed in the previous cell.
Identify the column that most clearly contains the sample groups, such as Case/Disease and Control/Healthy.

Available phenotype columns:
 - sample_id
 - title
 - geo_accession
 - status
 - submission_date
 - last_update_date
 - type
 - channel_count
 - source_name_ch1
 - organism_ch1
 - taxid_ch1
 - characteristics_ch1
 - molecule_ch1
 - extract_protocol_ch1
 - label_ch1
 - label_protocol_ch1
 - hyb_protocol
 - scan_protocol
 - description
 - data_processing
 - platform_id
 - contact_name
 - contact_email
 - contact_laboratory
 - contact_department
 - contact_institute
 - contact_address
 - contact_city
 - contact_zip/postal_code
 - contact_country
 - supplementary_file
 - series_id
 - data_row_count
 - characteristics_0
 - characteristics_1
 - characteristics_2

Enter the phenotype column name (example: characteristics_1): characteristics_1

Selected phenotype 

,geo_accession,description
0,GSM1426071,disease: Crohn's disease
1,GSM1426072,disease: Crohn's disease
2,GSM1426073,disease: Crohn's disease
3,GSM1426074,disease: Crohn's disease
4,GSM1426075,disease: Crohn's disease


In [14]:
# @title Step 10: Create a mapping between GSM IDs and descriptions (Just run the cell)
print("\nStep 10: Creating sample description mapping...")

# Create a mapping dictionary while retaining GSM IDs when a description is missing.
desc_mapping = {}
for gsm_id, description in zip(sample_info["geo_accession"], sample_info["description"]):
    if pd.isna(description) or not str(description).strip():
        desc_mapping[str(gsm_id)] = str(gsm_id)
    else:
        desc_mapping[str(gsm_id)] = str(description).strip()

print("Description mapping (first 5 items):")
for i, (key, value) in enumerate(desc_mapping.items()):
    print(f"{key}: {value}")
    if i >= 4:
        break


Step 8: Creating sample description mapping...
Description mapping (first 5 items):
GSM1426071: disease: Crohn's disease
GSM1426072: disease: Crohn's disease
GSM1426073: disease: Crohn's disease
GSM1426074: disease: Crohn's disease
GSM1426075: disease: Crohn's disease


In [15]:
# @title # Step 11: Replace GSM IDs with descriptions in expression_set (Just run the cell)
print("\nStep 11: Replacing GSM IDs with descriptions...")

original_cols = expression_set.columns.tolist()
new_cols = []

for col in original_cols:
    if col == "ID":
        new_cols.append(col)
    else:
        # Preserve the original behavior: repeated biological descriptions may create
        # repeated column names, which are later renamed manually to control/disease.
        new_cols.append(desc_mapping.get(str(col), col))

# Assign the complete column list directly so every repeated description is retained.
expression_set.columns = new_cols

print("Updated column names (first 20):")
print(expression_set.columns.tolist()[:20])

# Save the DataFrame as in the original workflow.
expression_set.to_csv("expression_set.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set.csv")


Step 9: Replacing GSM IDs with descriptions...
Updated column names (first 20):
['ID', "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", "disease: Crohn's disease", 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control', 'disease: control']


In [16]:
# @title Step 12: Check the selected phenotype column and its unique values
# This cell automatically uses the phenotype column selected in Step 9.

print("\nStep 12: Checking the selected phenotype column...")

# Make sure Step 9 has already been run
if "selected_description_column" not in globals():
    raise RuntimeError(
        "No phenotype column has been selected yet. "
        "Please run Step 9 first and select the column containing "
        "Case/Disease and Control/Healthy information."
    )

column_name = selected_description_column

if column_name in pheno_data.columns:
    unique_names = (
        pheno_data[column_name]
        .dropna()
        .map(flatten_metadata_value)
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    print(f"\nSelected phenotype column: {column_name}")
    print("\nUnique phenotype values found:")

    for i, value in enumerate(unique_names, start=1):
        print(f"{i}. {value}")

else:
    raise KeyError(
        f"Selected column '{column_name}' was not found in pheno_data."
    )


Step 12: Checking the selected phenotype column...

Selected phenotype column: characteristics_1

Unique phenotype values found:
1. disease: Crohn's disease
2. disease: control
3. disease: ulcerative colitis
4. disease: CD


In [18]:
# @title Step 13: Select and remove unwanted sample groups
# Only unique phenotype labels are displayed.
# Enter one or more labels separated by commas. Leave blank if nothing should be removed.

print("\nStep 13: Select unwanted sample groups for removal...")

# Use the unique phenotype labels identified in Step 12
if "unique_names" not in globals():
    raise RuntimeError(
        "Unique phenotype labels were not found. "
        "Please run Step 12 first."
    )

print("\nUnique sample groups available:")
for i, label in enumerate(unique_names, start=1):
    print(f"{i}. {label}")

print(
    "\nEnter the exact group label(s) you want to remove."
    "\nFor multiple groups, separate the labels with commas."
    "\nExample: disease: ulcerative colitis, disease: indeterminate colitis"
)

user_input = input("\nGroup(s) to remove: ").strip()

if user_input:

    columns_to_drop = [
        item.strip()
        for item in user_input.split(",")
        if item.strip()
    ]

    # Check which requested labels actually exist in expression_set
    existing_columns_to_drop = [
        label for label in columns_to_drop
        if label in expression_set.columns
    ]

    missing_labels = [
        label for label in columns_to_drop
        if label not in expression_set.columns
    ]

    if existing_columns_to_drop:
        expression_set.drop(
            columns=existing_columns_to_drop,
            inplace=True
        )

        print("\nRemoved sample group(s):")
        for label in existing_columns_to_drop:
            print(f" - {label}")

    else:
        print("\nNone of the entered groups were found. No samples were removed.")

    if missing_labels:
        print("\nGroup label(s) not found:")
        for label in missing_labels:
            print(f" - {label}")

else:
    print("\nNo group was selected. No samples were removed.")

# Save updated dataset
expression_set.to_csv(
    "expression_set_updated.csv",
    index=False
)

print("\nUpdated dataset saved as: expression_set_updated.csv")

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set_updated.csv")


Step 13: Select unwanted sample groups for removal...

Unique sample groups available:
1. disease: Crohn's disease
2. disease: control
3. disease: ulcerative colitis
4. disease: CD

Enter the exact group label(s) you want to remove.
For multiple groups, separate the labels with commas.
Example: disease: ulcerative colitis, disease: indeterminate colitis

Group(s) to remove: disease: ulcerative colitis

Removed sample group(s):
 - disease: ulcerative colitis

Updated dataset saved as: expression_set_updated.csv


In [19]:
# @title Step 14: Assign the retained sample groups as Disease/Case or Control/Healthy
# This cell automatically uses the sample groups retained after Step 13.
# No manual editing of the code is required.

print("\nStep 14: Assigning retained sample groups...")

# Make sure Step 12 was run
if "unique_names" not in globals():
    raise RuntimeError(
        "Unique phenotype groups were not found. "
        "Please run Step 12 and Step 13 first."
    )

# Identify only the phenotype groups that are still present after Step 13
remaining_groups = [
    group for group in unique_names
    if group in expression_set.columns
]

if not remaining_groups:
    raise ValueError(
        "No phenotype groups were found after Step 13. "
        "Please check the previous steps."
    )

print("\nRetained sample groups after Step 13:")
for i, group in enumerate(remaining_groups, start=1):
    sample_count = sum(str(col) == str(group) for col in expression_set.columns)
    print(f"{i}. {group}  (n = {sample_count})")

print(
    "\nFor each retained group, specify whether it should be treated as:\n"
    "  - Disease / Case\n"
    "  - Control / Healthy\n"
    "\nAccepted inputs:\n"
    "Disease group: disease, case\n"
    "Control group: control, healthy\n"
)

# Automatically build the rename mapping
rename_mapping = {}

for group in remaining_groups:

    while True:
        user_choice = input(
            f"\nHow should '{group}' be classified? "
            "[Disease/Case or Control/Healthy]: "
        ).strip().lower()

        if user_choice in ["disease", "case"]:
            rename_mapping[group] = "disease"
            break

        elif user_choice in ["control", "healthy"]:
            rename_mapping[group] = "control"
            break

        else:
            print(
                "Invalid input. Please enter one of the following: "
                "disease, case, control, or healthy."
            )

# Show the selected mapping before applying it
print("\nSelected group classification:")
for original_group, final_group in rename_mapping.items():
    print(f" - {original_group}  →  {final_group}")

# Rename phenotype-labelled sample columns
expression_set.rename(columns=rename_mapping, inplace=True)

# Summary
n_control = sum(str(col) == "control" for col in expression_set.columns)
n_disease = sum(str(col) == "disease" for col in expression_set.columns)

print("\nFinal sample-group summary:")
print("Number of control samples:", n_control)
print("Number of disease samples:", n_disease)

# Save the renamed dataset
expression_set.to_csv(
    "expression_set_renamed.csv",
    index=False
)

print("\nSaved as: expression_set_renamed.csv")

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("expression_set_renamed.csv")


Step 14: Assigning retained sample groups...

Retained sample groups after Step 13:
1. disease: Crohn's disease  (n = 8)
2. disease: control  (n = 22)
3. disease: CD  (n = 67)

For each retained group, specify whether it should be treated as:
  - Disease / Case
  - Control / Healthy

Accepted inputs:
Disease group: disease, case
Control group: control, healthy


How should 'disease: Crohn's disease' be classified? [Disease/Case or Control/Healthy]: disease

How should 'disease: control' be classified? [Disease/Case or Control/Healthy]: control

How should 'disease: CD' be classified? [Disease/Case or Control/Healthy]: disease

Selected group classification:
 - disease: Crohn's disease  →  disease
 - disease: control  →  control
 - disease: CD  →  disease

Final sample-group summary:
Number of control samples: 22
Number of disease samples: 75

Saved as: expression_set_renamed.csv


In [20]:
# @title # Step 15: Merge expression data with gene information (Just run the cell)
print("\nStep 15: Merging datasets...")

# Standardize probe-ID type only; the original left merge is retained.
expression_set["ID"] = expression_set["ID"].astype(str).str.strip()
gene_info["probe_id"] = gene_info["probe_id"].astype(str).str.strip()

final_data = pd.merge(
    expression_set,
    gene_info,
    left_on="ID",
    right_on="probe_id",
    how="left",
)

print("Merged final_data shape:", final_data.shape)
display(final_data.head())

# Save merged data
final_data.to_csv("final_data.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("final_data.csv")


Step 13: Merging datasets...
Merged final_data shape: (33252, 101)


,ID,disease,disease,disease,disease,disease,disease,disease,disease,control,...,control,control,control,control,control,control,control,probe_id,gene_symbol,gene_id
0,7892501,3.097244,4.459059,3.322385,4.184156,3.567690,4.434777,3.416537,3.286221,2.897307,...,3.404894,4.506227,2.637388,4.403679,3.310945,3.651002,3.141966,NaN,NaN,NaN
1,7892502,4.622620,4.866747,5.793462,5.652146,5.106890,5.180421,5.198954,5.169658,5.014645,...,5.476374,5.059218,4.862137,5.287762,5.010893,5.194320,5.077890,NaN,NaN,NaN
2,7892503,4.588315,4.635416,7.472632,6.024110,4.213697,4.001750,4.180109,4.076452,4.189199,...,4.759516,4.487999,4.315397,4.510607,4.947265,4.357883,4.655735,NaN,NaN,NaN
3,7892504,9.635181,8.897930,9.450976,8.800208,9.105323,9.350480,9.066997,8.958961,8.896822,...,9.003382,9.400929,9.672492,9.295140,9.452490,9.183286,9.279613,NaN,NaN,NaN
4,7892505,4.524358,4.131027,4.633418,4.228925,4.757025,4.703955,3.916718,4.552928,4.238462,...,4.337687,5.196698,4.058416,4.535763,4.855737,4.711060,4.523698,NaN,NaN,NaN


In [23]:
# @title Step 16: Finalize gene identifiers and save the processed dataset (Just run the cell)
# Replace ID with gene_symbol and delete the specified probe columns.

import pandas as pd
from google.colab import files

# Load the final_data.csv file
final_data = pd.read_csv("final_data.csv")

if "gene_symbol" not in final_data.columns:
    raise KeyError("gene_symbol was not created during probe annotation.")

# Retain the original replacement operation.
final_data["ID"] = final_data["gene_symbol"]

# Drop the unnecessary columns safely.
final_data.drop(columns=["probe_id", "gene_id"], inplace=True, errors="ignore")

# Save the modified DataFrame to a new CSV file
final_data.to_csv("modified_final_data.csv", index=False)

if DOWNLOAD_INTERMEDIATE_FILES:
    files.download("modified_final_data.csv")

In [24]:
# @title Step 17: Verify control and disease sample columns in the processed dataset (Just run the cell)

import pandas as pd

modified_final_data = pd.read_csv("modified_final_data.csv")

print(modified_final_data.head())

# CSV readers make repeated names unique as control, control.1, control.2, etc.
control_columns = [
    col for col in modified_final_data.columns
    if str(col) == "control" or str(col).startswith("control.")
]

disease_columns = [
    col for col in modified_final_data.columns
    if str(col) in {"disease", "case"}
    or str(col).startswith("disease.")
    or str(col).startswith("case.")
]

print("\nNumber of control sample columns:", len(control_columns))
print("Control columns:", control_columns)
print("\nNumber of disease sample columns:", len(disease_columns))
print("Disease columns:", disease_columns)

if control_columns:
    print("\nControl expression summary:")
    print(modified_final_data[control_columns].apply(pd.to_numeric, errors="coerce").stack().describe())

if disease_columns:
    print("\nDisease expression summary:")
    print(modified_final_data[disease_columns].apply(pd.to_numeric, errors="coerce").stack().describe())

    ID   disease  disease.1  disease.2  disease.3  disease.4  disease.5  \
0  NaN  3.097244   4.459059   3.322385   4.184156   3.567690   4.434777   
1  NaN  4.622620   4.866747   5.793462   5.652146   5.106890   5.180421   
2  NaN  4.588315   4.635416   7.472632   6.024110   4.213697   4.001750   
3  NaN  9.635181   8.897930   9.450976   8.800208   9.105323   9.350480   
4  NaN  4.524358   4.131027   4.633418   4.228925   4.757025   4.703955   

   disease.6  disease.7   control  ...  control.13  control.14  control.15  \
0   3.416537   3.286221  2.897307  ...    3.508979    2.937225    3.404894   
1   5.198954   5.169658  5.014645  ...    5.289645    4.914546    5.476374   
2   4.180109   4.076452  4.189199  ...    4.926381    4.039336    4.759516   
3   9.066997   8.958961  8.896822  ...    9.303847    9.013299    9.003382   
4   3.916718   4.552928  4.238462  ...    4.365907    3.488814    4.337687   

   control.16  control.17  control.18  control.19  control.20  control.21  \
0  

In [25]:
# @title Step 18: Finalize and reorder the processed dataset (Just run the cell)
import pandas as pd

# Load the modified CSV file
df = pd.read_csv("modified_final_data.csv")

# Keep a copy exactly as in the original workflow.
df_modified = df.copy()

# Remove accidental index columns while retaining the biological data.
unnamed_columns = [col for col in df_modified.columns if str(col).startswith("Unnamed:")]
if unnamed_columns:
    df_modified.drop(columns=unnamed_columns, inplace=True)

if "gene_symbol" not in df_modified.columns:
    if "ID" in df_modified.columns:
        df_modified.rename(columns={"ID": "gene_symbol"}, inplace=True)
    else:
        raise KeyError("Neither gene_symbol nor ID was found in modified_final_data.csv")

# Create the final DataFrame with gene_symbol as the first column and remove ID.
columns = ["gene_symbol"] + [
    col for col in df_modified.columns
    if col not in {"gene_symbol", "ID"}
]
df_final = df_modified[columns]

print(df_final.head())
print("Final reordered shape:", df_final.shape)

# Save to the same intermediate filename as the original workflow.
df_final.to_csv("modified_final_data.csv", index=False)

  gene_symbol   disease  disease.1  disease.2  disease.3  disease.4  \
0         NaN  3.097244   4.459059   3.322385   4.184156   3.567690   
1         NaN  4.622620   4.866747   5.793462   5.652146   5.106890   
2         NaN  4.588315   4.635416   7.472632   6.024110   4.213697   
3         NaN  9.635181   8.897930   9.450976   8.800208   9.105323   
4         NaN  4.524358   4.131027   4.633418   4.228925   4.757025   

   disease.5  disease.6  disease.7   control  ...  control.12  control.13  \
0   4.434777   3.416537   3.286221  2.897307  ...    3.068425    3.508979   
1   5.180421   5.198954   5.169658  5.014645  ...    4.957102    5.289645   
2   4.001750   4.180109   4.076452  4.189199  ...    4.373730    4.926381   
3   9.350480   9.066997   8.958961  8.896822  ...    8.757489    9.303847   
4   4.703955   3.916718   4.552928  4.238462  ...    5.027381    4.365907   

   control.14  control.15  control.16  control.17  control.18  control.19  \
0    2.937225    3.404894    4.50

In [26]:
# @title Step 19: Download out clean output file
# Load the modified CSV file
df = pd.read_csv("modified_final_data.csv")

# Drop rows where gene_symbol is blank or NaN, retaining the original final-cleaning rule.
df = df[
    df["gene_symbol"].notna()
    & df["gene_symbol"].astype(str).str.strip().ne("")
].copy()

# Dynamic filename prevents the dataset-name mismatch present in the original notebook.
final_output_filename = f"cleaned_modified_final_data {GSE_ID}.csv"

df.to_csv(final_output_filename, index=False)

print("Final processed file:", final_output_filename)
print("Final processed shape:", df.shape)
print("Gene-symbol column is first:", df.columns[0] == "gene_symbol")

# This is the only automatic download when DOWNLOAD_INTERMEDIATE_FILES=False.
files.download(final_output_filename)

Final processed file: cleaned_modified_final_data GSE75214.csv
Final processed shape: (25293, 98)
Gene-symbol column is first: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>